<a href="https://colab.research.google.com/github/SUSHMASURE11/Predictive-Analytics/blob/Customer_Sales_Prediction/Customer_Sales_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# 1. Generate Complex Synthetic Dataset
np.random.seed(42)
n = 2000

data = pd.DataFrame({
    'Age': np.random.randint(18, 70, n),
    'Gender': np.random.choice(['Male', 'Female'], n),
    'Income': np.random.uniform(15000, 200000, n),
    'MembershipYears': np.random.randint(0, 15, n),
    'PreferredChannel': np.random.choice(['Online', 'In-Store', 'App'], n),
    'VisitedWebsite': np.random.choice([0, 1], n),
    'PurchasedLast30Days': np.random.choice([0, 1], n),
    'Region': np.random.choice(['North', 'South', 'East', 'West'], n),
    'DiscountUsed': np.random.choice([0, 1], n)
})

# Target: Sales Amount (realistic logic + noise)
data['Sales'] = (
    data['Income'] * np.random.uniform(0.01, 0.05, n) +
    data['PurchasedLast30Days'] * 1200 +
    data['VisitedWebsite'] * 500 +
    data['DiscountUsed'] * 300 +
    data['MembershipYears'] * 50 +
    np.random.normal(0, 1000, n)  # noise
)

# 2. Features and Target
X = data.drop('Sales', axis=1)
y = data['Sales']

# 3. Column Types
num_features = ['Age', 'Income', 'MembershipYears']
cat_features = ['Gender', 'PreferredChannel', 'Region', 'VisitedWebsite', 'PurchasedLast30Days', 'DiscountUsed']

# 4. Preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(drop='first'), cat_features)
])

# 5. Pipeline with GradientBoostingRegressor
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(random_state=42))
])

# 6. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. GridSearch for Best Params
param_grid = {
    'regressor__n_estimators': [100],
    'regressor__max_depth': [3, 5],
    'regressor__learning_rate': [0.05, 0.1]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='r2')
grid.fit(X_train, y_train)
best_model = grid.best_estimator_

# 8. Evaluate Model
y_pred = best_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📊 Model Performance:")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R² Score:", round(r2, 4))
print("Best Parameters:", grid.best_params_)

# 9. Predict Sales for a New Customer
new_customer = pd.DataFrame([{
    'Age': 35,
    'Gender': 'Male',
    'Income': 90000,
    'MembershipYears': 4,
    'PreferredChannel': 'Online',
    'VisitedWebsite': 1,
    'PurchasedLast30Days': 1,
    'Region': 'South',
    'DiscountUsed': 1
}])

predicted_sales = best_model.predict(new_customer)[0]
print(f"\n🛍️ Predicted Sales for New Customer: ₹{predicted_sales:.2f}")


📊 Model Performance:
MAE : 1369.71
RMSE: 1716.82
R² Score: 0.4758
Best Parameters: {'regressor__learning_rate': 0.05, 'regressor__max_depth': 3, 'regressor__n_estimators': 100}

🛍️ Predicted Sales for New Customer: ₹4592.54
